## 1. Data

In [108]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import keras
from keras import layers, models
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from keras_hub.layers import TransformerDecoder, TransformerEncoder, TokenAndPositionEmbedding
import sentencepiece as spm

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import kagglehub

In [ ]:
# Can probably be commented out if you have no problems with GPU / CUDA!!!
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))
tf.config.optimizer.set_jit(False)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

2.22.0-dev0+selfbuilt
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [110]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-2")

def load_lines(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.readlines()


train_lines = load_lines(wikitext2_path, "wiki.train.tokens")
val_lines   = load_lines(wikitext2_path, "wiki.valid.tokens")
test_lines  = load_lines(wikitext2_path, "wiki.test.tokens")

## 2. Data Cleaning

In [111]:
def clean_line(text_line):
    text_line = re.sub(r"^\s*=+\s*[^=\n]+?\s*=+\s*$", "", text_line)
    text_line = re.sub(r"\s+", " ", text_line)
    return text_line.strip()


train_lines_clean = [clean_line(text_line) for text_line in train_lines]
val_lines_clean   = [clean_line(text_line) for text_line in val_lines]
test_lines_clean  = [clean_line(text_line) for text_line in test_lines]

train_lines_clean = [line for line in train_lines_clean if line]
val_lines_clean   = [line for line in val_lines_clean if line]
test_lines_clean  = [line for line in test_lines_clean if line]

## 3. Subword Tokenizer

In [112]:
with open("wikitext_train.txt", "w", encoding="utf-8") as f:
    for line in train_lines_clean:
        f.write(line + "\n")

In [ ]:
vocab_size_limit = 8000


for f in ["spm_wikitext.model", "spm_wikitext.vocab"]:
    if os.path.exists(f):
        os.remove(f)

spm.SentencePieceTrainer.train(
    input="wikitext_train.txt",
    model_prefix="spm_wikitext",
    vocab_size=vocab_size_limit,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    input_sentence_size=500000,
    shuffle_input_sentence=True,
)

sp = spm.SentencePieceProcessor()
sp.load("spm_wikitext.model")

True

In [114]:
eos = sp.eos_id()

train_sequences = sp.encode(train_lines_clean, out_type=int)
val_sequences   = sp.encode(val_lines_clean, out_type=int)
test_sequences  = sp.encode(test_lines_clean, out_type=int)

In [115]:
train_sequences = [seq + [eos] for seq in train_sequences]
val_sequences   = [seq + [eos] for seq in val_sequences]
test_sequences  = [seq + [eos] for seq in test_sequences]

train_sequence = [tok for seq in train_sequences for tok in seq]
val_sequence   = [tok for seq in val_sequences for tok in seq]
test_sequence  = [tok for seq in test_sequences for tok in seq]

vocab_size = sp.get_piece_size()

print("Train tokens:", len(train_sequence))
print("Val tokens:", len(val_sequence))
print("Test tokens:", len(test_sequence))
print("Vocab size:", vocab_size)

Train tokens: 2733490
Val tokens: 293973
Test tokens: 337222
Vocab size: 8000


## 4. Dataset and Hyperparameters

In [ ]:
# Hyperparameters
batch_size = 64
train_stride = 8
val_stride = 16
test_stride = 16
seq_len = 128

train_tokens = np.array(train_sequence, dtype=np.int32)
val_tokens   = np.array(val_sequence, dtype=np.int32)
test_tokens  = np.array(test_sequence, dtype=np.int32)

def make_dataset(tokens, seq_len, batch_size, shuffle=False, stride=64):
    tokens = np.asarray(tokens, dtype=np.int32)

    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=tokens,
        targets=None,
        sequence_length=seq_len + 1,
        sequence_stride=stride,
        batch_size=batch_size,
        shuffle=shuffle,
    )

    ds = ds.map(lambda window: (window[:, :-1], window[:, 1:]))
    return ds.prefetch(tf.data.AUTOTUNE)


# Put .take(n) at the end of the functions if training is taking too long!!!
# Where n is the number of samples
train_ds = make_dataset(
    train_tokens,
    seq_len,
    batch_size,
    shuffle=True,
    stride=train_stride
)

val_ds = make_dataset(
    val_tokens,
    seq_len,
    batch_size,
    stride=val_stride
)

test_ds = make_dataset(
    test_tokens,
    seq_len,
    batch_size,
    stride=test_stride
)

In [118]:
print("characters:", len(train_lines_clean))
print("tokens:", len(train_tokens))
print("steps:", tf.data.experimental.cardinality(train_ds).numpy())

characters: 23138
tokens: 2733490
steps: 5339


## 5. Model

In [ ]:
embedding_dim = 256
embed_multiplier = 2
num_heads = 4
decoder_amount = 2
dropout_rate = 0.2


inputs = keras.Input(shape=(None,), dtype="int64", name="inputs")
x = TokenAndPositionEmbedding(
    vocabulary_size=vocab_size,
    sequence_length=seq_len,
    embedding_dim=embedding_dim,
    mask_zero=True,
)(inputs)

for _ in range(decoder_amount):
    x = TransformerDecoder(
        intermediate_dim=embedding_dim * embed_multiplier,
        num_heads=num_heads,
        dropout=dropout_rate,
    )(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(vocab_size)(x) # don't add softmax! SparceCategoricalCrossentropy calculates it internally!!!

transformer_model = keras.Model(inputs, outputs, name="decoder_only_transformer")

In [121]:
transformer_model.summary()

Model: "decoder_only_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_4  │ (None, None, 256)      │     2,080,768 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_8           │ (None, None, 256)      │       527,104 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_9           │ (None, None, 256)      │       527,104 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, None, 8000)     │     2,056,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,190,976 (19.80 MB)

 Trainable params: 5,190,976 (19.80 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Training

In [122]:
USE_EARLY_STOPPING = True


def make_callbacks(ckpt_path, use_early_stopping=USE_EARLY_STOPPING):
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor="val_loss",
            save_best_only=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    if use_early_stopping:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=2,
                min_delta=0.001,
                start_from_epoch=3,
                restore_best_weights=True,
                verbose=1,
            )
        )

    return callbacks

In [123]:
class Perplexity(keras.metrics.Metric):
    def __init__(self, name="perplexity", **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_tracker = keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        loss = keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=True
        )

        self.loss_tracker.update_state(loss, sample_weight=sample_weight)

    def result(self):
        return tf.exp(self.loss_tracker.result())

    def reset_state(self):
        self.loss_tracker.reset_state()

In [124]:
epochs = 20

transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4
        ),
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top_5_accuracy"),
        Perplexity(),
    ],
    jit_compile=False
)

transformer_history = transformer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=make_callbacks("best_model.keras"),
)

Epoch 1/20
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 622s 116ms/step - accuracy: 0.2653 - loss: 4.4232 - perplexity: 83.3612 - top_5_accuracy: 0.4587 - val_accuracy: 0.3450 - val_loss: 3.8281 - val_perplexity: 45.9765 - val_top_5_accuracy: 0.5343 - learning_rate: 3.0000e-04
Epoch 2/20
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 630s 118ms/step - accuracy: 0.3238 - loss: 3.7604 - perplexity: 42.9640 - top_5_accuracy: 0.5344 - val_accuracy: 0.3566 - val_loss: 3.7622 - val_perplexity: 43.0447 - val_top_5_accuracy: 0.5465 - learning_rate: 3.0000e-04
Epoch 3/20
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 635s 119ms/step - accuracy: 0.3443 - loss: 3.5854 - perplexity: 36.0669 - top_5_accuracy: 0.5567 - val_accuracy: 0.3611 - val_loss: 3.7478 - val_perplexity: 42.4277 - val_top_5_accuracy: 0.5507 - learning_rate: 3.0000e-04
Epoch 4/20
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 634s 119ms/step - accuracy: 0.3560 - loss: 3.4908 - perplexity: 32.8117 - top_5_accuracy: 0.5688 - val_accuracy: 0.3627 - val_loss: 3.7520 - val_perplexity: 42.6071 -

In [125]:
for loss in transformer_history.history["val_loss"]:
    print(np.exp(loss))

45.976525934381634
43.04467382312199
42.42772411181036
42.607107168236084
42.95083196669828
42.8424220840442


## 7. Sampling

In [ ]:
def sample_probs(logits, temperature=1.0):
    logits = np.asarray(logits).astype("float64")
    logits = logits / temperature
    logits = logits - np.max(logits)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits)

In [126]:
# Not in regex, because the model performs better with the context of special symbols
# in sentences
BAD_PIECES = {
    "<pad>", "<unk>", "<s>", "</s>",
    ".", ",", ";", ":", '"',
    "▁.", "▁,", "▁;", "▁:", "▁\"",
    "<", ">", "▁<", "▁>",
    "(", ")", "▁(", "▁)",
    "__&", "__/",

    # WikiText artifacts
    "@-@", "▁@-@",
    "-", "▁-",
    "—", "▁—",
}

BAD_IDS = {
    sp.pad_id(),
    sp.unk_id(),
    sp.bos_id(),
    sp.eos_id(),
}

STOP_PIECES = {
    "▁and", "▁in", "▁of", "▁for", "▁was", "▁is", "▁the", "▁a", "▁to", "▁by"
}

STOP_IDS = {
    sp.piece_to_id(piece)
    for piece in STOP_PIECES
    if sp.piece_to_id(piece) != sp.unk_id()
}

for piece in BAD_PIECES:
    piece_id = sp.piece_to_id(piece)
    if piece_id != sp.unk_id():
        BAD_IDS.add(piece_id)


def encode_prompt(text):
    seq = sp.encode(text, out_type=int)
    seq = seq[-seq_len:]
    return np.array([seq], dtype=np.int32)


def predict_top_n(text, model, top_n=5, temperature=0.8, filter_bad=True):
    seq = encode_prompt(text)

    logits = model.predict(seq, verbose=0)[0, -1]

    if filter_bad:
        for bad_id in BAD_IDS:
            if 0 <= bad_id < len(logits):
                logits[bad_id] = -1e9


        for stop_id in STOP_IDS:
            if 0 <= stop_id < len(logits):
                logits[stop_id] -= 2.0

    probs = sample_probs(logits, temperature)
    sorted_indices = np.argsort(probs)[::-1]

    results = []
    for i in sorted_indices:
        decoded = sp.decode([int(i)])
        results.append((decoded, round(float(probs[i]) * 100, 2)))

        if len(results) == top_n:
            break

    return results


def sample_next_token_id(text, model, temperature=0.8, top_k=30):
    seq = encode_prompt(text)

    logits = model.predict(seq, verbose=0)[0, -1]

    for bad_id in BAD_IDS:
        if 0 <= bad_id < len(logits):
            logits[bad_id] = -1e9
    
    for stop_id in STOP_IDS:
        if 0 <= stop_id < len(logits):
            logits[stop_id] -= 2.0

    probs = sample_probs(logits, temperature)

    top_indices = np.argsort(probs)[::-1][:top_k]
    top_probs = probs[top_indices]

    if np.sum(top_probs) == 0:
        return sp.unk_id()

    top_probs = top_probs / np.sum(top_probs)

    sampled_id = np.random.choice(top_indices, p=top_probs)
    return int(sampled_id)


def generate_text(seed_text, model, num_tokens=40, temperature=0.8, top_k=30):
    generated_ids = sp.encode(seed_text, out_type=int)

    for _ in range(num_tokens):
        current_text = sp.decode(generated_ids)

        next_id = sample_next_token_id(
            current_text,
            model,
            temperature=temperature,
            top_k=top_k,
        )

        generated_ids.append(next_id)

    return sp.decode(generated_ids)

In [ ]:
seed = "the capital of france is"

print("\n=== Transformer: Top-5 seuraavaa sanaa ===")
for piece, pct in predict_top_n(seed, transformer_model, top_n=5):
    print(f"  {piece:20s} {pct:.2f}%")

print("\n=== Transformer generoitu teksti ===")
print(generate_text(
    seed,
    transformer_model,
    num_tokens=30,
    temperature=0.6, # puts more rare instances of words in the pool the higher the temperature is
    top_k=40
))


=== Transformer: Top-5 seuraavaa sanaa ===
                       12.04%
  now                  4.48%
  not                  3.84%
  located              1.62%
  used                 1.48%

=== Transformer generoitu teksti ===
the capital of france is now known as the          Sitriuc 's    reigning      his
